# DermaLens AI · ConvNeXt V2 Tiny · Real-only · SMOKE / FULL

**Dataset:** `shanmukhadattaboda/facial-dermatology-image-dataset`, six classes, frozen train/validation/test splits.  
**Model:** `timm/convnextv2_tiny.fcmae_ft_in1k` — authentic pretrained FCMAE + ImageNet classification checkpoint (28.6 M parameters).  
**Experiment:** train on **real** data only. No synthetic images, no patient-level claims, no test-based model selection.

### One-click mode switch
- `RUN_MODE = "SMOKE"` (default): few images per class, one epoch for each progressive stage, checks full pipeline and export. **Metrics cannot be reported as research results.**
- `RUN_MODE = "FULL"`: entire original dataset, proper multi-epoch progressive training and saved artifacts.
- Optional complete fine-tuning is **off in FULL** and tested for one epoch in SMOKE to cover its code path.

### Method: Progressive block-wise fine-tuning
1. Train **head only**.
2. Train **head + stages.3** (last ConvNeXt V2 stage).
3. Train **head + stages.2 + stages.3**.
4. Optionally fine-tune **all** blocks with very small layer-wise LRs.

Each stage begins from its own **best validation checkpoint**; the best overall checkpoint is chosen by **validation macro-F1**, breaking ties with validation loss. Hold-out test evaluation happens only after selection.  

**Pretraining license:** The publisher identifies this pretrained checkpoint as **CC-BY-NC-4.0**; respect its noncommercial restrictions. Notebook is for research, not diagnosis.  
**Verification limits:** A tiny smoke pass cannot guarantee full-data integrity, cloud availability or Kaggle runtime compatibility. Perform FULL as a separate saved version.

## Cell 1 — Dependencies, imports, runtime checks

`torch`, `torchvision`, `Pillow`, `scikit-learn`, `numpy`, `pandas` and `matplotlib` come with standard Kaggle Python images. Use `timm==1.0.20` for ConvNeXt V2 if missing. **Do not upgrade torch/torchvision here**; that can break Kaggle CUDA bindings. This cell validates installed `timm` architecture support. Pretrained weights are obtained in Cell 6 (Internet must be enabled on first run).

In [ ]:
import os, sys, json, csv, time, random, hashlib, logging, traceback, shutil, zipfile, math
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from contextlib import nullcontext
import importlib.util, subprocess

# Kaggle typically includes timm; install only if absent to avoid disturbing its CUDA environment.
if importlib.util.find_spec("timm") is None:
    print("Installing timm==1.0.20 (Internet required on a fresh Kaggle runtime).")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm==1.0.20"])

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageFile, UnidentifiedImageError
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
import timm

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_recall_fscore_support
)

ImageFile.LOAD_TRUNCATED_IMAGES = False
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "TorchVision:", __import__("torchvision").__version__, "timm:", timm.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
else:
    print("WARNING: No GPU found. FULL training is not recommended on CPU.")

## Cell 2 — Central configuration (SMOKE / FULL)

Only change **`RUN_MODE`** after a successful smoke run. The rest is a reproducible starting configuration; optional full-network fine-tuning is disabled in FULL to keep compute manageable. SMOKE additionally exercises that branch. Each mode writes a separate directory and ZIP so one cannot overwrite the other.

Expected dataset slug: `facial-dermatology-image-dataset`, attached via Kaggle **Add Input**. The notebook locates the true split root even if there is one extra containing folder.

In [ ]:
from dataclasses import dataclass, asdict

# <<< CHANGE ONLY THIS LINE AFTER THE SMOKE RUN PASSES >>>
RUN_MODE = "SMOKE"  # Options: "SMOKE" or "FULL"
if RUN_MODE not in {"SMOKE", "FULL"}:
    raise ValueError("RUN_MODE must be exactly SMOKE or FULL")

@dataclass(frozen=True)
class Config:
    dataset_slug: str = "facial-dermatology-image-dataset"
    output_name: str = "convnextv2_tiny_real_progressive_v1"
    model_name: str = "convnextv2_tiny"
    imagenet_weights: str = "timm:convnextv2_tiny.fcmae_ft_in1k"
    image_size: int = 224
    batch_size: int = 16
    num_workers: int = 2
    epochs_head: int = 4
    epochs_stage3: int = 8
    epochs_stages2_3: int = 8
    full_fine_tune: bool = False
    epochs_full: int = 5
    patience: int = 4
    min_epochs_per_stage: int = 2
    use_amp: bool = True
    weight_decay: float = 1e-4
    grad_clip_norm: float = 1.0
    use_class_weights: bool = True
    verify_full_sha256: bool = True
    max_suspicious_images_report: int = 100
    seed: int = SEED
    # SMOKE mode only. Sample counts are capped by availability in each class.
    smoke_train_per_class: int = 6
    smoke_val_per_class: int = 3
    smoke_test_per_class: int = 2
    smoke_batch_size: int = 4
    smoke_epochs_per_stage: int = 1
    smoke_test_optional_full_stage: bool = True

CFG = Config()
SMOKE = (RUN_MODE == "SMOKE")
CLASSES = ["Acne", "Fungal_infection", "Hyper_Pigmentation", "Normal_Skin", "Skin_cancer", "Vitiligo"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASSES)}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
RUN_OUTPUT_NAME = CFG.output_name + ("_SMOKE" if SMOKE else "_FULL")
OUT = OUTPUT_ROOT / RUN_OUTPUT_NAME
for folder in (OUT, OUT / "checkpoints", OUT / "plots", OUT / "reports"):
    folder.mkdir(parents=True, exist_ok=True)
(OUT / "config.json").write_text(json.dumps({**asdict(CFG), "classes": CLASSES, "class_to_idx": CLASS_TO_IDX, "run_mode": RUN_MODE,
                                                "effective_batch_size": CFG.smoke_batch_size if SMOKE else CFG.batch_size}, indent=2), encoding="utf-8")
(OUT / "class_to_idx.json").write_text(json.dumps(CLASS_TO_IDX, indent=2), encoding="utf-8")

LOGGER = logging.getLogger("dermalens_convnextv2_tiny")
LOGGER.setLevel(logging.INFO)
LOGGER.propagate = False
for old_handler in list(LOGGER.handlers):
    LOGGER.removeHandler(old_handler)
fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(OUT / "training.log", encoding="utf-8")
file_handler.setFormatter(fmt)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(fmt)
LOGGER.addHandler(file_handler)
LOGGER.addHandler(stream_handler)

def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, default=str, allow_nan=False), encoding="utf-8")

LOGGER.info("Configuration saved: %s", OUT / "config.json")
LOGGER.info("RUN MODE: %s (SMOKE results are NOT scientifically valid)", RUN_MODE)
LOGGER.info("Full fine-tuning enabled in FULL mode: %s", CFG.full_fine_tune)

## Cell 3 — Find the dataset automatically

The notebook searches the attached Kaggle dataset for a *single* folder containing `train/`, `validation/` and `test/` and rejects ambiguous/missing layouts. It never alters source files.

In [ ]:
def resolve_dataset_root(base: Path) -> Path:
    # Support Kaggle's flat and nested dataset mounts.
    search_roots = [
        base,
        Path("/kaggle/input/datasets/shanmukhadattaboda/facial-dermatology-image-dataset"),
        Path("/kaggle/input/facial-dermatology-image-dataset"),
    ]
    found = set()
    for root in search_roots:
        if not root.is_dir():
            continue
        if all((root / s).is_dir() for s in ("train", "validation", "test")):
            found.add(root.resolve())
        for train_dir in root.rglob("train"):
            if train_dir.is_dir() and all(
                (train_dir.parent / s).is_dir() for s in ("train", "validation", "test")
            ):
                found.add(train_dir.parent.resolve())
    if len(found) != 1:
        visible = sorted(str(p) for p in Path("/kaggle/input").glob("*"))
        raise FileNotFoundError(
            "Attach the FINAL split Kaggle dataset "
            "'shanmukhadattaboda/facial-dermatology-image-dataset', "
            "not the old unsplit 'facial-skin-conditions-real-image-collection'. "
            f"Matching split roots: {sorted(map(str, found))}. "
            f"Mounted /kaggle/input entries: {visible}"
        )
    return next(iter(found))

INPUT_BASE = Path("/kaggle/input") / CFG.dataset_slug
DATA_ROOT = resolve_dataset_root(INPUT_BASE)
SPLIT_DIRS = {s: DATA_ROOT / s for s in ("train", "validation", "test")}
LOGGER.info("Real dataset root: %s", DATA_ROOT)
print("Found split folders:", {k: str(v) for k, v in SPLIT_DIRS.items()})


## Cell 4 — Audit image files and preserve immutable splits

Verify six class folders in every split. `SMOKE` samples a fixed-seed subset of each original split. `FULL` checks every file (PIL verify, RGB decode, dark/uniform warnings, SHA-256 cross-split exact duplicates); near-duplicates/patient overlap need external provenance checks. All counts and warnings are saved for later review.

In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

records = []
issues = []
all_split_counts = {}
random_selector = random.Random(CFG.seed)
smoke_caps = {"train": CFG.smoke_train_per_class,
              "validation": CFG.smoke_val_per_class,
              "test": CFG.smoke_test_per_class}
for split, split_root in SPLIT_DIRS.items():
    expected = set(CLASSES)
    actual = {p.name for p in split_root.iterdir() if p.is_dir()}
    if actual != expected:
        raise RuntimeError(f"Wrong class directories in {split}: missing={expected-actual}, extra={actual-expected}")
    for label in CLASSES:
        image_paths = sorted(
            p for p in (split_root / label).rglob("*")
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS
        )
        if not image_paths:
            raise RuntimeError(f"No images found under {split_root / label}")
        all_split_counts[f"{split}/{label}"] = len(image_paths)
        if SMOKE:
            # Stratified fixed-seed sample, without changing any original split.
            k = min(smoke_caps[split], len(image_paths))
            if k <= 0:
                raise RuntimeError(f"SMOKE sampling found no images in {split}/{label}")
            image_paths = sorted(random_selector.sample(image_paths, k))
        for path in image_paths:
            rel = str(path.relative_to(DATA_ROOT))
            try:
                with Image.open(path) as im:
                    im.verify()
                with Image.open(path) as im:
                    im = ImageOps.exif_transpose(im).convert("RGB")
                    width, height = im.size
                    if width < 32 or height < 32:
                        issues.append({"path": rel, "issue": "very_small_image", "width": width, "height": height})
                    gray = np.asarray(im.resize((32, 32)).convert("L"), dtype=np.float32)
                    if float(gray.std()) < 3 or float(gray.mean()) < 8:
                        issues.append({"path": rel, "issue": "near_uniform_or_dark", "width": width, "height": height})
                digest = sha256_file(path) if CFG.verify_full_sha256 else "not_computed"
                records.append({
                    "split": split, "class_name": label, "class_id": CLASS_TO_IDX[label],
                    "relative_path": rel, "width": width, "height": height,
                    "file_size_bytes": path.stat().st_size, "sha256": digest
                })
            except (OSError, ValueError, UnidentifiedImageError) as exc:
                issues.append({"path": rel, "issue": "corrupt_or_unreadable", "error": str(exc)})
    LOGGER.info("Validated %s images: %d", split, sum(r["split"] == split for r in records))

manifest = pd.DataFrame.from_records(records)
manifest.to_csv(OUT / "reports" / "dataset_manifest.csv", index=False)
pd.DataFrame(issues, columns=["path", "issue", "width", "height", "error"]).to_csv(OUT / "reports" / "image_warnings.csv", index=False)

bad = [x for x in issues if x["issue"] == "corrupt_or_unreadable"]
if bad:
    raise RuntimeError(f"Found {len(bad)} corrupt images. See reports/image_warnings.csv. Fix the source dataset before training.")

if CFG.verify_full_sha256:
    overlap = (
        manifest.groupby("sha256", sort=False)
        .agg(n_splits=("split", "nunique"), count=("split", "size"))
        .reset_index()
    )
    leaked_hashes = set(overlap.loc[overlap.n_splits > 1, "sha256"])
    leakage = manifest[manifest.sha256.isin(leaked_hashes)].sort_values(["sha256", "split"])
    leakage.to_csv(OUT / "reports" / "cross_split_exact_duplicates.csv", index=False)
    if leaked_hashes:
        raise RuntimeError(f"Found {len(leaked_hashes)} exact cross-split duplicate hashes. See cross_split_exact_duplicates.csv")
    LOGGER.info("SHA-256 cross-split audit passed for %s files only.", "SAMPLED" if SMOKE else "ALL")
else:
    LOGGER.warning("Full SHA-256 disabled: exact cross-split duplicate check NOT performed.")

summary = (manifest.groupby(["split", "class_name"]).size()
           .unstack(fill_value=0).reindex(index=["train", "validation", "test"], columns=CLASSES))
summary.to_csv(OUT / "reports" / "split_class_counts.csv")
dataset_fingerprint = hashlib.sha256(
    manifest.sort_values("relative_path")[["relative_path", "sha256"]]
    .to_csv(index=False).encode("utf-8")
).hexdigest()
save_json(OUT / "reports" / "dataset_audit.json", {
    "root": str(DATA_ROOT), "fingerprint": dataset_fingerprint,
    "run_mode": RUN_MODE, "audit_scope": "sample_only" if SMOKE else "entire_dataset",
    "original_class_file_counts": all_split_counts,
    "total_images": int(len(manifest)), "split_sizes": {s: int(summary.loc[s].sum()) for s in summary.index},
    "class_names": CLASSES, "sha256_checked": CFG.verify_full_sha256,
    "exact_split_overlap_count": 0 if CFG.verify_full_sha256 else None,
    "image_warning_count": len(issues),
    "limitations": ("SMOKE verifies ONLY a selected subset; not a full dataset audit. " if SMOKE else "")
                   + "Exact hashes do not detect visually similar/augmented duplicates or patient overlap."
})
print(summary.to_string())
LOGGER.info("Dataset fingerprint: %s", dataset_fingerprint)
LOGGER.warning("Suspect/very dark images (review, not auto-removed): %d", len(issues))
if SMOKE:
    LOGGER.warning("SMOKE: audited %d sampled images, NOT the full dataset. Results are NOT for publication.", len(manifest))
else:
    LOGGER.info("FULL: audited %d images. Near-duplicate/patient checks need separate provenance.", len(manifest))


## Cell 5 — Dataset and medically conservative transforms

Convert to RGB, apply EXIF correction, preserve aspect ratio using bicubic padding to 224×224, normalize using the actual pretrained model's ImageNet statistics. Only training uses mild horizontal flips/rotations; validation and test are deterministic. A saved JSON records the exact procedure so external evaluation can reproduce it.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
PAD_COLOR = (128, 128, 128)

class AspectPad:
    def __init__(self, size: int, color=PAD_COLOR):
        self.size = size
        self.color = color
    def __call__(self, image):
        return ImageOps.pad(image, (self.size, self.size), method=Image.Resampling.BICUBIC, color=self.color, centering=(0.5, 0.5))

train_transform = transforms.Compose([
    AspectPad(CFG.image_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=7, interpolation=InterpolationMode.BILINEAR, fill=PAD_COLOR),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
valid_transform = transforms.Compose([
    AspectPad(CFG.image_size),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class FixedImageDataset(Dataset):
    def __init__(self, items, root: Path, transform):
        self.items = items.reset_index(drop=True)
        self.root = root
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, index):
        row = self.items.iloc[index]
        image_path = self.root / row["relative_path"]
        with Image.open(image_path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            tensor = self.transform(image)
        return tensor, int(row["class_id"]), str(row["relative_path"])

train_ds = FixedImageDataset(manifest[manifest.split == "train"], DATA_ROOT, train_transform)
val_ds = FixedImageDataset(manifest[manifest.split == "validation"], DATA_ROOT, valid_transform)
test_ds = FixedImageDataset(manifest[manifest.split == "test"], DATA_ROOT, valid_transform)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

pin_memory = DEVICE.type == "cuda"

def make_loader(ds, *, shuffle=False):
    generator = torch.Generator().manual_seed(CFG.seed)
    return DataLoader(
        ds, batch_size=CFG.smoke_batch_size if SMOKE else CFG.batch_size, shuffle=shuffle, num_workers=CFG.num_workers,
        pin_memory=pin_memory, persistent_workers=(CFG.num_workers > 0),
        worker_init_fn=seed_worker, generator=generator
    )

train_loader = make_loader(train_ds, shuffle=True)
val_loader = make_loader(val_ds)
test_loader = make_loader(test_ds)
assert len(train_ds) and len(val_ds) and len(test_ds)
xs, ys, paths = next(iter(train_loader))
assert xs.ndim == 4 and xs.shape[1:] == (3, CFG.image_size, CFG.image_size)
assert ys.min() >= 0 and ys.max() < len(CLASSES)
LOGGER.info("Data loading sanity check passed: %s", tuple(xs.shape))

## Cell 6 — Authentic pretrained ConvNeXt V2 Tiny

Load **`convnextv2_tiny.fcmae_ft_in1k`** via timm with authentic weights, *before* replacing the 1000-class head by a six-class head. If downloading fails, stop with an informative message — **do not silently train from random initialization**. Verify model output tensor shape.

Source: [timm model card](https://huggingface.co/timm/convnextv2_tiny.fcmae_ft_in1k).

In [ ]:
PRETRAINED_MODEL_NAME = "convnextv2_tiny.fcmae_ft_in1k"
try:
    # IMPORTANT: load real weights before changing classifier; no silent random fallback.
    model = timm.create_model(PRETRAINED_MODEL_NAME, pretrained=True)
except Exception as exc:
    raise RuntimeError(
        "Could not load authentic ConvNeXt V2 Tiny pretrained weights from timm/Hugging Face. "
        "Enable Kaggle Internet or attach a verified cached checkpoint. "
        f"Original error: {exc}"
    ) from exc
if not hasattr(model, "head") or not hasattr(model, "stages") or len(model.stages) != 4:
    raise RuntimeError("Unexpected timm ConvNeXt V2 structure: expected head and 4 stages")
model.reset_classifier(num_classes=len(CLASSES))
# timm initializes a fresh classifier after reset_classifier.
model = model.to(DEVICE)
param_total = sum(p.numel() for p in model.parameters())
LOGGER.info("ConvNeXt V2 Tiny loaded: %d parameters, pretrained=%s", param_total, PRETRAINED_MODEL_NAME)
if param_total < 20_000_000:
    raise RuntimeError("Unexpected parameter count: model could be the wrong architecture")
print("Model:", PRETRAINED_MODEL_NAME, "total_params:", param_total)

## Cell 7 — Progressive stages and optimizer groups

Train the classification head, then last stage, then the last two stages. Optional full fine-tuning uses small discriminative LRs for early layers. Verify that *every* trainable parameter belongs to exactly one LR group, preventing accidental no-op or unexpectedly full-model training. Class weights are calculated from the training split alone.

In [ ]:
train_counts = np.array([int((manifest.split.eq("train") & manifest.class_name.eq(c)).sum()) for c in CLASSES], dtype=np.float64)
if np.any(train_counts <= 0):
    raise RuntimeError("Every class must have training examples")
raw_weights = np.sqrt(train_counts.sum() / (len(CLASSES) * train_counts))
raw_weights = np.clip(raw_weights, 0.5, 2.0)
raw_weights /= raw_weights.mean()
class_weights = torch.tensor(raw_weights, dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights if CFG.use_class_weights else None)
LOGGER.info("Train counts: %s; loss weights: %s", train_counts.astype(int).tolist(), raw_weights.round(4).tolist())

STAGE_SPECS = [
    {"name": "head", "epochs": CFG.epochs_head, "lrs": {"head": 8e-4}},
    {"name": "stage3", "epochs": CFG.epochs_stage3, "lrs": {"stage3": 2e-5, "head": 1e-4}},
    {"name": "stages2_3", "epochs": CFG.epochs_stages2_3,
     "lrs": {"stage2": 7e-6, "stage3": 1.5e-5, "head": 5e-5}},
]
if CFG.full_fine_tune or (SMOKE and CFG.smoke_test_optional_full_stage):
    STAGE_SPECS.append({
        "name": "full", "epochs": CFG.epochs_full,
        "lrs": {"stem": 5e-7, "stage0": 7e-7, "stage1": 1.5e-6,
                "stage2": 4e-6, "stage3": 8e-6, "head": 3e-5}
    })
if SMOKE:
    STAGE_SPECS = [{**stage, "epochs": CFG.smoke_epochs_per_stage} for stage in STAGE_SPECS]
LOGGER.info("Effective %s stage schedule: %s", RUN_MODE, [(s["name"], s["epochs"]) for s in STAGE_SPECS])

def group_for_parameter(name: str) -> str:
    if name.startswith("head."):
        return "head"
    if name.startswith("stem."):
        return "stem"
    for i in range(4):
        if name.startswith(f"stages.{i}."):
            return f"stage{i}"
    raise RuntimeError(f"Unrecognized ConvNeXtV2 parameter (check timm version): {name}")

def prepare_stage(model, stage):
    for p in model.parameters():
        p.requires_grad_(False)
    buckets = defaultdict(list)
    for name, p in model.named_parameters():
        group = group_for_parameter(name)
        if group in stage["lrs"]:
            p.requires_grad_(True)
            buckets[group].append(p)
    if set(buckets) != set(stage["lrs"]):
        raise RuntimeError(f"Missing LR groups: {set(stage['lrs']) - set(buckets)}")
    optimizer = torch.optim.AdamW(
        [{"params": buckets[group], "lr": lr, "group_name": group}
         for group, lr in stage["lrs"].items()],
        weight_decay=CFG.weight_decay
    )
    count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    LOGGER.info("Stage %s: trainable parameters = %d; LR groups = %s", stage["name"], count, stage["lrs"])
    return optimizer

def freeze_bn_running_stats(module):
    if isinstance(module, nn.modules.batchnorm._BatchNorm):
        module.eval()

def metric_dict(y_true, y_pred, loss):
    return {
        "loss": float(loss),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=list(range(len(CLASSES))), average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

# Quick architecture contract test (forward only). A separate smoke training tests backward.
model.eval()
with torch.inference_mode():
    logits = model(xs[:min(2, len(xs))].to(DEVICE))
assert logits.shape == (min(2, len(xs)), len(CLASSES))
del xs, ys, paths, logits
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
LOGGER.info("Model/label contract test passed.")

## Cell 8 — Train/evaluate functions, AMP, history, safe checkpoints

One epoch records cross-entropy loss, accuracy, macro-F1, balanced accuracy, per-image class probabilities (validation/test) and timing. Training uses AdamW and gradient clipping with AMP on CUDA. Checkpoints include **all model weights**, classes, preprocessor, model name, validation score, dataset fingerprint, and stage. Saves are atomic (temporary file + rename).

In [ ]:
AMP_ENABLED = CFG.use_amp and DEVICE.type == "cuda"
SCALER = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
HISTORY_PATH = OUT / "reports" / "history.csv"
history_rows = []

def amp_context():
    return torch.amp.autocast(device_type="cuda", enabled=True) if AMP_ENABLED else nullcontext()

def run_epoch(model, loader, *, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    if is_train:
        model.apply(freeze_bn_running_stats)
    running_loss, seen = 0.0, 0
    truths, guesses, prediction_rows = [], [], []
    context = torch.enable_grad() if is_train else torch.inference_mode()
    with context:
        for batch_x, batch_y, batch_paths in loader:
            batch_x = batch_x.to(DEVICE, non_blocking=pin_memory)
            batch_y = batch_y.to(DEVICE, non_blocking=pin_memory)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
            with amp_context():
                batch_logits = model(batch_x)
                loss = criterion(batch_logits, batch_y)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite loss detected; stopped to protect checkpoints")
            if is_train:
                SCALER.scale(loss).backward()
                SCALER.unscale_(optimizer)
                nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), CFG.grad_clip_norm)
                SCALER.step(optimizer)
                SCALER.update()
            probs = torch.softmax(batch_logits.detach().float(), dim=1)
            predictions = probs.argmax(dim=1)
            y = batch_y.detach().cpu().tolist()
            yhat = predictions.detach().cpu().tolist()
            truths.extend(y)
            guesses.extend(yhat)
            if not is_train:
                all_probs = probs.detach().cpu().numpy()
                for row_path, y_true, y_pred, probability_vector in zip(batch_paths, y, yhat, all_probs):
                    row = {"relative_path": row_path, "true_id": int(y_true), "pred_id": int(y_pred),
                           "true_label": CLASSES[y_true], "pred_label": CLASSES[y_pred],
                           "confidence": float(probability_vector[y_pred])}
                    row.update({f"prob_{c}": float(probability_vector[i]) for i, c in enumerate(CLASSES)})
                    prediction_rows.append(row)
            batch_size = len(batch_y)
            running_loss += float(loss.detach().item()) * batch_size
            seen += batch_size
    assert seen > 0
    metrics = metric_dict(truths, guesses, running_loss / seen)
    return metrics, prediction_rows

def save_checkpoint(file_path, model, stage, epoch, val_metrics):
    payload = {
        "model_name": CFG.model_name, "run_mode": RUN_MODE,
        "checkpoint_purpose": "pipeline_verification_only" if SMOKE else "research_training",
        "weights_origin": CFG.imagenet_weights,
        "timm_model_name": PRETRAINED_MODEL_NAME,
        "architecture": "timm.convnextv2_tiny",
        "model_state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
        "num_classes": len(CLASSES), "classes": CLASSES, "class_to_idx": CLASS_TO_IDX,
        "preprocessing": {"input_size": CFG.image_size, "resize": "aspect_ratio_pad",
                          "pad_rgb": list(PAD_COLOR), "mean": list(IMAGENET_MEAN),
                          "std": list(IMAGENET_STD), "exif_transpose": True},
        "stage": stage, "stage_epoch": int(epoch), "validation": val_metrics,
        "dataset_fingerprint": dataset_fingerprint,
        "random_seed": CFG.seed,
        "torch_version": str(torch.__version__),
        "created_utc": datetime.now(timezone.utc).isoformat()
    }
    temp = Path(str(file_path) + ".tmp")
    torch.save(payload, temp)
    os.replace(temp, file_path)

# Compare validation macro-F1 first; on exact ties, prefer lower validation loss.
def better(candidate, incumbent, eps=1e-8):
    if incumbent is None:
        return True
    if candidate["macro_f1"] > incumbent["macro_f1"] + eps:
        return True
    return (abs(candidate["macro_f1"] - incumbent["macro_f1"]) <= eps
            and candidate["loss"] < incumbent["loss"] - eps)

## Cell 9 — Fully automated staged training

For each stage, run the configured epochs, save history CSV continuously and best checkpoints on val macro-F1, apply cosine LR decay, early-stop when appropriate, then **reload that stage's best checkpoint before the next stage**. Final model selection never uses the test set. Exceptions in the training loop are saved to `error_traceback.txt` and `run_status.json`.

In [ ]:
def safe_load_checkpoint(file_path, map_location="cpu"):
    try:
        return torch.load(file_path, map_location=map_location, weights_only=True)
    except TypeError as exc:
        raise RuntimeError("This notebook requires PyTorch with torch.load(weights_only=True).") from exc

def run_all_stages():
    global_best = None
    global_path = OUT / "checkpoints" / "best_model.pt"
    stage_results = []
    for stage in STAGE_SPECS:
        name = stage["name"]
        LOGGER.info("=== BEGIN STAGE: %s | max epochs: %d ===", name, stage["epochs"])
        optimizer = prepare_stage(model, stage)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, stage["epochs"]), eta_min=0)
        stage_best = None
        stage_path = OUT / "checkpoints" / f"best_{name}.pt"
        stagnant = 0
        for epoch in range(1, stage["epochs"] + 1):
            tick = time.perf_counter()
            train_metrics, _ = run_epoch(model, train_loader, optimizer=optimizer)
            validation_metrics, _ = run_epoch(model, val_loader)
            row = {"stage": name, "epoch_in_stage": epoch,
                   "train_loss": train_metrics["loss"], "train_accuracy": train_metrics["accuracy"],
                   "train_macro_f1": train_metrics["macro_f1"],
                   "val_loss": validation_metrics["loss"], "val_accuracy": validation_metrics["accuracy"],
                   "val_macro_f1": validation_metrics["macro_f1"],
                   "val_balanced_accuracy": validation_metrics["balanced_accuracy"],
                   "seconds": round(time.perf_counter() - tick, 2)}
            row.update({f"lr_{group['group_name']}": float(group["lr"]) for group in optimizer.param_groups})
            history_rows.append(row)
            pd.DataFrame(history_rows).to_csv(HISTORY_PATH, index=False)
            LOGGER.info("%s epoch %02d/%02d | train_loss=%.4f val_loss=%.4f val_macro_f1=%.4f val_acc=%.4f sec=%.1f",
                        name, epoch, stage["epochs"], train_metrics["loss"], validation_metrics["loss"],
                        validation_metrics["macro_f1"], validation_metrics["accuracy"], row["seconds"])
            if better(validation_metrics, stage_best):
                stage_best = dict(validation_metrics)
                save_checkpoint(stage_path, model, name, epoch, validation_metrics)
                stagnant = 0
            else:
                stagnant += 1
            if better(validation_metrics, global_best):
                global_best = dict(validation_metrics)
                save_checkpoint(global_path, model, name, epoch, validation_metrics)
                LOGGER.info("Updated global best at %s epoch %d: val_macro_f1=%.4f", name, epoch, validation_metrics["macro_f1"])
            scheduler.step()
            if epoch >= CFG.min_epochs_per_stage and stagnant >= CFG.patience:
                LOGGER.info("Early stopping %s after epoch %d", name, epoch)
                break
        if stage_best is None:
            raise RuntimeError(f"Stage {name} completed without a checkpoint")
        stage_results.append({"stage": name, "best_val": stage_best, "checkpoint": str(stage_path), "completed_epochs": epoch})
        save_json(OUT / "reports" / "stage_results.json", stage_results)
        model.load_state_dict(safe_load_checkpoint(stage_path, DEVICE)["model_state_dict"], strict=True)
        LOGGER.info("Reloaded best checkpoint from stage %s before next stage.", name)
    if not global_path.is_file():
        raise RuntimeError("No globally selected model checkpoint found")
    return global_path, stage_results

save_json(OUT / "run_status.json", {"status": "smoke_running" if SMOKE else "full_running", "run_mode": RUN_MODE, "started_utc": datetime.now(timezone.utc).isoformat()})
try:
    BEST_PATH, STAGE_RESULTS = run_all_stages()
    save_json(OUT / "run_status.json", {"status": "smoke_trained_not_yet_evaluated" if SMOKE else "full_trained_not_yet_evaluated", "run_mode": RUN_MODE, "best_checkpoint": str(BEST_PATH),
                                        "updated_utc": datetime.now(timezone.utc).isoformat()})
except Exception as exc:
    tb = traceback.format_exc()
    (OUT / "error_traceback.txt").write_text(tb, encoding="utf-8")
    save_json(OUT / "run_status.json", {"status": "failed_during_training", "run_mode": RUN_MODE, "error": str(exc)})
    LOGGER.exception("Training failed")
    raise

## Cell 10 — Reload selected checkpoint and evaluate locked test

Reload the validation-selected checkpoint. Save validation and test probability CSVs, classification report JSON/TXT, confusion matrix CSV, and final metrics JSON. SMOKE results are clearly marked **non-reportable**. The test is only used after training and checkpoint selection.

In [ ]:
best_payload = safe_load_checkpoint(BEST_PATH, DEVICE)
assert best_payload["class_to_idx"] == CLASS_TO_IDX
assert best_payload["dataset_fingerprint"] == dataset_fingerprint
model.load_state_dict(best_payload["model_state_dict"], strict=True)
model.to(DEVICE).eval()

selected_val_metrics, selected_val_rows = run_epoch(model, val_loader)
val_predictions = pd.DataFrame(selected_val_rows)
val_predictions.to_csv(OUT / "reports" / "validation_predictions.csv", index=False)

if SMOKE:
    LOGGER.warning("SMOKE TEST metrics are pipeline diagnostics, NOT held-out research results!")
else:
    LOGGER.info("FULL: held-out test evaluation begins; do not change training settings from test feedback.")
test_metrics, test_rows = run_epoch(model, test_loader)
test_predictions = pd.DataFrame(test_rows)
test_predictions.to_csv(OUT / "reports" / "test_predictions.csv", index=False)

truths = test_predictions.true_id.astype(int).tolist()
preds = test_predictions.pred_id.astype(int).tolist()
report_dict = classification_report(truths, preds, labels=list(range(len(CLASSES))),
                                    target_names=CLASSES, zero_division=0, output_dict=True)
report_text = classification_report(truths, preds, labels=list(range(len(CLASSES))),
                                    target_names=CLASSES, zero_division=0)
save_json(OUT / "reports" / "test_classification_report.json", report_dict)
(OUT / "reports" / "test_classification_report.txt").write_text(report_text, encoding="utf-8")
cm = confusion_matrix(truths, preds, labels=list(range(len(CLASSES))))
pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(OUT / "reports" / "test_confusion_matrix.csv")

metrics_output = {
    "model": CFG.model_name, "training_data": "real_only",
    "run_mode": RUN_MODE, "evaluation_scope": "smoke_subset_debug_only" if SMOKE else "full_holdout_test",
    "publishable": not SMOKE,
    "selection_metric": "validation_macro_f1",
    "selected_stage": best_payload["stage"], "selected_epoch_within_stage": best_payload["stage_epoch"],
    "best_validation_at_selection": best_payload["validation"],
    "validation_after_reload": selected_val_metrics,
    "test": test_metrics,
    "test_classification_report": report_dict,
    "dataset_fingerprint": dataset_fingerprint,
    "test_samples": len(truths), "classes": CLASSES,
    "warning": ("SMOKE SUBSET METRICS ARE INVALID FOR SCIENTIFIC REPORTING. " if SMOKE else "")
               + "Research-only image classification. Test set must not be used for model selection."
}
save_json(OUT / "reports" / "final_metrics.json", metrics_output)
LOGGER.info("SELECTED: %s epoch %d", best_payload["stage"], best_payload["stage_epoch"])
LOGGER.info("FINAL TEST: %s", test_metrics)
print("\n=== " + ("SMOKE / NON-REPORTABLE" if SMOKE else "FULL HOLDOUT") + " TEST REPORT ===\n", report_text)

## Cell 11 — Save plots for comparison

Store learning curves and raw/normalized confusion matrices as PNGs. These plots can later be compared with ResNet-50, DINOv2, VMamba, PanDerm, and EfficientNet-B0. Note that SMOKE curves are debugging only.

In [ ]:
hist = pd.read_csv(HISTORY_PATH)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for stage_name, rows in hist.groupby("stage", sort=False):
    step = rows.index.to_numpy() + 1
    axes[0].plot(step, rows.train_loss, marker=".", label=f"{stage_name} train")
    axes[0].plot(step, rows.val_loss, marker=".", linestyle="--", label=f"{stage_name} val")
    axes[1].plot(step, rows.val_macro_f1, marker="o", label=stage_name)
axes[0].set(title="Training and validation loss", xlabel="Global epoch", ylabel="Cross-entropy loss")
axes[1].set(title="Validation macro-F1", xlabel="Global epoch", ylabel="Macro-F1", ylim=(0, 1.02))
for ax in axes:
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "plots" / "learning_curves.png", dpi=160)
plt.close(fig)

for normalized in (False, True):
    plot_cm = cm.astype(float)
    if normalized:
        denom = plot_cm.sum(axis=1, keepdims=True)
        plot_cm = np.divide(plot_cm, denom, out=np.zeros_like(plot_cm), where=denom > 0)
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(plot_cm, cmap="Blues", vmin=0, vmax=1 if normalized else None)
    ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=35, ha="right")
    ax.set_yticks(range(len(CLASSES)), CLASSES)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title("Held-out test confusion matrix" + (" (row-normalized)" if normalized else " (counts)"))
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            text_value = f"{plot_cm[i, j]:.2f}" if normalized else str(int(plot_cm[i, j]))
            ax.text(j, i, text_value, ha="center", va="center", fontsize=9,
                    color="white" if plot_cm[i, j] > (0.5 if normalized else cm.max()/2) else "black")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(OUT / "plots" / ("confusion_normalized.png" if normalized else "confusion_counts.png"), dpi=160)
    plt.close(fig)
LOGGER.info("Saved learning curves and confusion matrices.")

## Cell 12 — Portable inference script and reload consistency test

A separate `inference_example.py` reconstructs the **same timm ConvNeXt V2 architecture with `pretrained=False`** and loads the six-class checkpoint, so other Kaggle notebooks can use it without downloading pretrained weights. The code compares independently reloaded CPU predictions against the notebook validation predictions (within floating-point tolerance).

In [ ]:
PREPROCESS_META = best_payload["preprocessing"]
save_json(OUT / "preprocessing.json", PREPROCESS_META)
inference_script = r'''"""Reusable DermaLens ConvNeXt V2 Tiny inference (research only).

Usage:
  python inference_example.py --checkpoint /path/checkpoints/best_model.pt --image /path/photo.jpg

Install timm if not available. Saved classifier weights are complete; no pretrained download is needed.
"""
import argparse, json
import torch
import timm
from torchvision import transforms
from PIL import Image, ImageOps


def load_classifier(checkpoint_path, device="cpu"):
    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    labels = payload["classes"]
    name = payload["timm_model_name"]
    model = timm.create_model(name, pretrained=False, num_classes=len(labels))
    model.load_state_dict(payload["model_state_dict"], strict=True)
    model.to(device).eval()
    return model, payload


def make_preprocess(meta):
    size = int(meta["input_size"])
    color = tuple(meta["pad_rgb"])
    def pad(image):
        return ImageOps.pad(image, (size, size), method=Image.Resampling.BICUBIC, color=color)
    return transforms.Compose([pad, transforms.ToTensor(), transforms.Normalize(meta["mean"], meta["std"])])

@torch.inference_mode()
def predict(checkpoint_path, image_path, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model, payload = load_classifier(checkpoint_path, device=device)
    preprocessing = make_preprocess(payload["preprocessing"])
    with Image.open(image_path) as im:
        tensor = preprocessing(ImageOps.exif_transpose(im).convert("RGB")).unsqueeze(0).to(device)
    probabilities = torch.softmax(model(tensor), dim=1)[0].cpu().tolist()
    return {"prediction": payload["classes"][int(max(range(len(probabilities)), key=probabilities.__getitem__))],
            "probabilities": {name: float(p) for name, p in zip(payload["classes"], probabilities)},
            "caution": "Research only — not for clinical diagnosis"}

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--image", required=True)
    args = parser.parse_args()
    print(json.dumps(predict(args.checkpoint, args.image), indent=2))
'''
(OUT / "inference_example.py").write_text(inference_script, encoding="utf-8")
compile(inference_script, str(OUT / "inference_example.py"), "exec")
reuse_metadata = {
    "artifact_schema_version": "dermalens-classifier-v1",
    "model": CFG.model_name, "run_mode": RUN_MODE,
    "publishable": not SMOKE,
    "checkpoint_relative_path": "checkpoints/best_model.pt",
    "class_map_relative_path": "class_to_idx.json",
    "preprocess_relative_path": "preprocessing.json",
    "metrics_relative_path": "reports/final_metrics.json",
    "test_predictions_relative_path": "reports/test_predictions.csv",
    "dataset_fingerprint": dataset_fingerprint,
    "architecture": "timm.create_model('convnextv2_tiny.fcmae_ft_in1k', pretrained=False, num_classes=6)",
    "checkpoint_load": "torch.load(path, map_location='cpu', weights_only=True)",
    "test_set_locked": True,
    "note": "Use checkpoint and bundled inference_example.py for downstream evaluation; no Internet needed."
}
save_json(OUT / "artifact_manifest.json", reuse_metadata)
independent_ns = {"__name__": "dermalens_reuse_check"}
exec(compile(inference_script, "inference_example.py", "exec"), independent_ns)
probe_image = DATA_ROOT / str(val_predictions.iloc[0]["relative_path"])
reuse_out = independent_ns["predict"](str(BEST_PATH), str(probe_image), device="cpu")
expected = val_predictions.iloc[0]
for class_name in CLASSES:
    if abs(float(reuse_out["probabilities"][class_name]) - float(expected[f"prob_{class_name}"])) > 2e-3:
        raise AssertionError(f"Independent inference mismatch for {class_name}")
save_json(OUT / "reports" / "inference_reload_check.json", {
    "passed": True, "mode": RUN_MODE, "image": str(expected["relative_path"]),
    "predicted_class": reuse_out["prediction"], "probability_tolerance": 0.002,
    "note": "Independent CPU model reload and preprocessing agree with validation predictions."
})
LOGGER.info("Independent inference script and checkpoint reload verified.")

## Cell 13 — Package run artifacts for downstream evaluation

Save environment versions, package manifest, training log, reports, checkpoints and inference script, then ZIP everything into a self-contained artifact. Use this saved Kaggle output as Input to the future evaluation/comparison notebooks. A `smoke_passed` status means the pipeline worked on a small sample — **it does not validate full data or produce publishable metrics**.

In [ ]:
package_info = {
    "python": sys.version.split()[0],
    "torch": str(torch.__version__),
    "torchvision": __import__("torchvision").__version__,
    "timm": timm.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pillow": Image.__version__ if hasattr(Image, "__version__") else "unknown",
    "sklearn": __import__("sklearn").__version__,
    "cuda_version": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "stage_count": len(STAGE_SPECS), "run_mode": RUN_MODE,
    "audit_scope": "subset_only" if SMOKE else "full_dataset",
    "best_stage": best_payload["stage"],
    "best_stage_epoch": best_payload["stage_epoch"],
    "dataset_fingerprint": dataset_fingerprint,
}
save_json(OUT / "environment.json", package_info)

readme = f"""# DermaLens ConvNeXt V2 Tiny Real-Only Progressive Fine-Tuning

## MODE: {RUN_MODE}
- **RESEARCH STATUS:** {"SMOKE CHECK ONLY — NOT REPORTABLE" if SMOKE else "FULL REAL-DATA TRAINING"}
- To run the complete experiment, change `RUN_MODE = "SMOKE"` to `RUN_MODE = "FULL"` in the central settings cell and Save & Run All.
- SMOKE audits only sampled files and gives non-publishable debugging metrics.

## Checkpoint
- `checkpoints/best_model.pt`: selected by validation macro-F1 (ties: lower validation loss).
- `checkpoints/best_<stage>.pt`: best checkpoint from each completed stage.
- Selected stage: **{best_payload['stage']}**, epoch **{best_payload['stage_epoch']}**.
- Dataset fingerprint: `{dataset_fingerprint}`.

## Inputs and outputs
- Input: frozen `train/`, `validation/`, `test/` Kaggle real dataset.
- `config.json`: reproducible training configuration.
- `preprocessing.json`, `class_to_idx.json`: loader contract.
- `artifact_manifest.json`: downstream evaluation adapter contract.
- `reports/dataset_manifest.csv`, `reports/split_class_counts.csv`: immutable input audit.
- `reports/history.csv`, `reports/stage_results.json`: complete training history.
- `reports/final_metrics.json`, `reports/test_classification_report.*`: final evaluations.
- `reports/test_predictions.csv`: all per-image test probabilities.
- `plots/*.png`: training curves and confusion matrices.
- `training.log`: complete application log.
- `inference_example.py`: inference without re-downloading pretrained weights.

## Loading the artifact in another Kaggle notebook
1. Add this **notebook's saved output** as Input.
2. Locate `checkpoints/best_model.pt` under `/kaggle/input/`.
3. Use `inference_example.py` or reconstruct `timm.create_model('convnextv2_tiny.fcmae_ft_in1k', pretrained=False, num_classes=6)`.
4. `torch.load(path, map_location='cpu', weights_only=True)` and use the embedded `preprocessing` metadata.

## Research precautions
- This is research software, **not a validated clinical diagnostic system**.
- The held-out test was evaluated only once for the finalized run; do not tune to the test results.
- Exact SHA-256 cross-split matching cannot detect near-duplicates or same-patient overlap.
- Default run uses one seed; additional independent seeds are needed for uncertainty estimates.
- Results cannot substantiate patient-level generalization without patient-disjoint identifiers.
"""
(OUT / "README.md").write_text(readme, encoding="utf-8")
zip_path = OUTPUT_ROOT / f"{RUN_OUTPUT_NAME}_artifact.zip"
# Finalize metadata, then archive outputs in one pass.
save_json(OUT / "run_status.json", {
    "status": "smoke_passed" if SMOKE else "full_completed", "run_mode": RUN_MODE,
    "publishable": not SMOKE, "updated_utc": datetime.now(timezone.utc).isoformat(),
    "checkpoint": str(BEST_PATH), "zip": str(zip_path),
    "selected_stage": best_payload["stage"], "val_macro_f1": float(best_payload["validation"]["macro_f1"]),
    "test_macro_f1": float(test_metrics["macro_f1"])
})
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, mode="w", compression=zipfile.ZIP_DEFLATED, compresslevel=3,
                     allowZip64=True) as bundle:
    for path in sorted(OUT.rglob("*")):
        if path.is_file() and not path.name.endswith(".tmp"):
            bundle.write(path, arcname=str(path.relative_to(OUT)))

print("\n=== " + RUN_MODE + " RUN COMPLETED ===")
print("Selected stage:", best_payload["stage"])
print("Selected validation macro-F1:", round(best_payload["validation"]["macro_f1"], 4))
print("Final test macro-F1:", round(test_metrics["macro_f1"], 4))
print("Best checkpoint:", BEST_PATH)
print("Reusable artifact ZIP:", zip_path, "size MiB:", round(zip_path.stat().st_size / 2**20, 1))
LOGGER.info("%s pipeline completed; outputs exported.", RUN_MODE)
if SMOKE:
    print("SMOKE PASSED: code paths executed on a tiny subset. NOT a full dataset audit or research result.")
    print('NEXT: Change RUN_MODE = "SMOKE" to RUN_MODE = "FULL" in Cell 2, then Save & Run All.')

## What to do after running in Kaggle

1. Add [Facial Dermatology Image Dataset](https://www.kaggle.com/datasets/shanmukhadattaboda/facial-dermatology-image-dataset/data) using **Add Input**.
2. Use **GPU T4**, enable Internet (for timm/checkpoint on first run), keep `RUN_MODE = "SMOKE"`, and press **Save Version → Save & Run All**.
3. Verify `SMOKE PASSED` and `run_status.json: status=smoke_passed`. Inspect errors if any. Check that the real pretrained checkpoint was loaded.
4. Change only `RUN_MODE = "FULL"`. **Save Version → Save & Run All** again with the entire fixed dataset.
5. Download `convnextv2_tiny_real_progressive_v1_FULL_artifact.zip` or add the completed Kaggle notebook's output as Input to the central evaluation notebook.

**Limitations:** Training length and memory use depend on Kaggle T4 availability and input sizes. Full runs may still fail from data, network, CUDA runtime, or session limits even after a smoke pass. Keep separate complete training artifacts for each model; compare results on the same untouched real test split. Never treat smoke metrics as scientific evidence.